# MedAgent: 05. Multimodal Deep Learning & Cross-Attention Fusion
## Integrating ECG Signals, Chest X-Rays, and Clinical Patient Data

This notebook demonstrates the multimodal fusion architecture:
1. Signal Feature Extraction (1D CNN / Transformer $\to 256\text{-d}$ embedding).
2. Vision Feature Extraction (DenseNet121 $\to 1024\text{-d} \to 256\text{-d}$ embedding).
3. Tabular Clinical Context ($64\text{-d} \to 256\text{-d}$ embedding).
4. Multi-Head Cross-Attention Transformer fusion for unified patient representation.
5. Overall Patient Risk Stratification (**Low**, **Moderate**, **High/Critical**).
6. Explainability: Grad-CAM heatmap visualization on Chest X-rays.

In [ ]:
import sys
from pathlib import Path

# Robust project root determination whether cwd is MedAgent or MedAgent/notebooks
current_dir = Path.cwd().resolve()
if (current_dir / "src").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").exists():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ensure venv site-packages are accessible even if run from a generic kernel
VENV_SITE = PROJECT_ROOT / "venv" / "Lib" / "site-packages"
if VENV_SITE.exists() and str(VENV_SITE) not in sys.path:
    sys.path.insert(0, str(VENV_SITE))

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Compute Device: {device}")

### 1. Define Model Architectures
Let's define the 1D CNN Signal Encoder, Vision Encoder, and Cross-Attention Fusion Transformer.

In [ ]:
class ECG1DCNN(nn.Module):
    def __init__(self, in_channels=1, latent_dim=256, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.proj = nn.Linear(128, latent_dim)
        self.classifier = nn.Linear(latent_dim, num_classes)

    def forward(self, x):
        feat = self.features(x).squeeze(-1)
        latent = F.relu(self.proj(feat))
        logits = self.classifier(latent)
        return logits, latent


class MultimodalTransformerFusion(nn.Module):
    def __init__(self, embed_dim=256, nheads=8, num_layers=3, num_risk_classes=3):
        super().__init__()
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nheads, dim_feedforward=512, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.risk_head = nn.Linear(embed_dim, num_risk_classes)

    def forward(self, ecg_embed, vision_embed, tabular_embed):
        # Shape: (B, 1, D) for each token
        B = ecg_embed.size(0)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        
        tokens = torch.stack([ecg_embed, vision_embed, tabular_embed], dim=1)  # (B, 3, D)
        all_tokens = torch.cat([cls_tokens, tokens], dim=1)  # (B, 4, D)
        
        out = self.transformer(all_tokens)
        cls_out = out[:, 0, :]  # Output at CLS token
        risk_logits = self.risk_head(cls_out)
        return risk_logits, cls_out

print("Model Architectures Initialized Successfully.")

### 2. Multimodal Forward Pass Simulation

In [ ]:
ecg_model = ECG1DCNN(latent_dim=256).to(device)
fusion_model = MultimodalTransformerFusion(embed_dim=256).to(device)

# Simulated inputs
batch_size = 4
dummy_ecg = torch.randn(batch_size, 1, 260).to(device)
dummy_vision = torch.randn(batch_size, 256).to(device)
dummy_tabular = torch.randn(batch_size, 256).to(device)

ecg_logits, ecg_latent = ecg_model(dummy_ecg)
risk_logits, patient_rep = fusion_model(ecg_latent, dummy_vision, dummy_tabular)

risk_probs = F.softmax(risk_logits, dim=-1)
risk_classes = ["Low Risk", "Moderate Risk", "High / Critical Risk"]

print(f"ECG Logits Shape: {ecg_logits.shape}")
print(f"Patient Unified Representation Shape: {patient_rep.shape}")
print(f"Multimodal Risk Probabilities:\n{risk_probs.detach().cpu().numpy()}")
for i in range(batch_size):
    pred_risk = risk_classes[torch.argmax(risk_probs[i]).item()]
    print(f"  Patient {i+1}: Classified as '{pred_risk}' (Confidence: {risk_probs[i].max().item()*100:.1f}%)")

### 3. Explainability with Grad-CAM Visualization

In [ ]:
gradcam_sample_path = PROJECT_ROOT / "data" / "processed" / "images" / "gradcam_sample.png"

if gradcam_sample_path.exists():
    img_gradcam = Image.open(gradcam_sample_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img_gradcam)
    plt.title("Sample Grad-CAM Radiograph Localization (Cardiomegaly/Effusion)", fontsize=13, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Grad-CAM sample image not found.")